# MNIST MLP3 — AdamW baseline versus ECS probe-loss TraceWall

Strict paired experiment: clean `AdamW` baseline versus the identical optimizer plus a task-directed TraceWall correction. At each correction the current matrices are truncated to the self-consistent ECS, a **rotating subset of the MNIST training set** defines cross-entropy, its gradient is projected into that ECS, and Armijo backtracking adds only a loss-decreasing component. The official test set is evaluation-only (`official_test_set_used_for_optimization=False`).

Protocol: `784 -> 512 -> 512 -> 10` ReLU MLP3; seeds `1337, 2027, 31415`; 20 epochs; one-epoch linear warmup; cosine decay to 5% of peak; one correction per epoch; 512 rotating probe examples; per-epoch WeightWatcher alpha/ERG diagnostics; 95% Student-t intervals; complete paired checkpoints.

In [ ]:
from pathlib import Path
import importlib, os, subprocess, sys
try:
    importlib.import_module("weightwatcher")
except ImportError:
    subprocess.check_call([sys.executable,"-m","pip","install","-q","weightwatcher>=0.7.7"])
ROOT=None
for p in [Path.cwd(),*Path.cwd().parents]:
    c=p/"optimizers"/"ecs_probe_loss_trace_wall"
    if (c/"ecs_trace_wall").is_dir(): ROOT=p.resolve(); OPT_ROOT=c.resolve(); break
    if (p/"ecs_trace_wall").is_dir() and p.name=="ecs_probe_loss_trace_wall": OPT_ROOT=p.resolve(); ROOT=p.parents[1].resolve(); break
if ROOT is None: raise RuntimeError("Run from a CalculatedContent/rg_optimizers clone")
for p in (OPT_ROOT,ROOT/"baseline"):
    if str(p) not in sys.path: sys.path.insert(0,str(p))
def artifact_dir(variable,default):
    p=Path(os.environ.get(variable,default)).expanduser()
    if not p.is_absolute(): p=Path.cwd()/p
    p=p.resolve(); p.mkdir(parents=True,exist_ok=True); return p
RUN_ROOT=artifact_dir("RG_TRACE_WALL_RUN_ROOT",OPT_ROOT/"runs")
DATA_DIR=artifact_dir("RG_TRACE_WALL_DATA_DIR",OPT_ROOT/"data")
print(ROOT,RUN_ROOT,DATA_DIR,sep="\n")

In [ ]:
from IPython.display import display
import numpy as np, pandas as pd
from ecs_trace_wall import BaseOptimizerConfig,ExperimentConfig,TraceWallConfig,plot_all,run_paired_experiment
pd.set_option("display.max_columns",None)
BASE_OPTIMIZER=BaseOptimizerConfig.adamw_baseline()
TRACE_WALL=TraceWallConfig(parameter_names=("fc1.weight","fc2.weight","fc3.weight"),probe_batch_size=256,probe_batches_per_correction=2,correction_to_base_step_ratio=0.25,minimum_weight_fraction=1e-5,maximum_weight_fraction=2.5e-3,projection_mode="core",min_ecs_rank=2,normalization_gamma=0.0,svd_device="cpu",use_backtracking=True,maximum_backtracking_steps=7,strict=True)
CONFIG=ExperimentConfig(optimizer=BASE_OPTIMIZER,trace_wall=TRACE_WALL,seeds=(1337, 2027, 31415),epochs=20,batch_size=128,num_workers=0,gradient_clip_norm=1.0,corrections_per_epoch=1,train_eval_max_batches=None,measure_weightwatcher=True,require_weightwatcher=True,save_epoch_checkpoints=True)
CONFIG.validate(); RUN_DIR=RUN_ROOT/BASE_OPTIMIZER.name; PLOT_DIR=RUN_DIR/"plots"
display(pd.json_normalize(CONFIG.to_dict(),sep=".").T.rename(columns={0:"value"}))

In [ ]:
result=run_paired_experiment(CONFIG,data_dir=DATA_DIR,output_dir=RUN_DIR,progress=True)
figures=plot_all(result,output_dir=PLOT_DIR,show=True)
print("saved",RUN_DIR,"figures",len(figures))

In [ ]:
display(result.performance.loc[result.performance.epoch.eq(CONFIG.epochs)].sort_values(["seed","arm"]))
display(result.performance_summary.loc[result.performance_summary.epoch.eq(CONFIG.epochs)].sort_values(["metric","arm"]))
display(result.spectral_summary.loc[result.spectral_summary.metric.isin(["ecs_rank","ecs_trace_log_per_eval","alpha","ERG_gap"])].sort_values(["metric","layer","arm","epoch"]))
events=result.corrections.drop_duplicates(["seed","global_step"]).sort_values(["seed","global_step"])
display(events[["seed","epoch","global_step","probe_loss_before","probe_loss_after","applied","line_search_scale"]])
display(result.correction_summary.sort_values(["parameter_name","seed"]))

In [ ]:
assert result.manifest["official_test_set_used_for_optimization"] is False
assert result.manifest["probe_source"]=="rotating subset of the MNIST training set"
assert result.manifest["probe_examples_per_correction"]==512 and result.manifest["corrections_per_epoch"]==1
for r in result.manifest["seed_runs"]:
    assert r["initial_checksum"]==r["baseline_initial_checksum"]==r["trace_wall_initial_checksum"]
    assert r["baseline_global_step"]==r["trace_wall_global_step"]
lr=result.performance.pivot_table(index=["seed","epoch"],columns="arm",values="learning_rate",aggfunc="first")
assert np.allclose(lr.baseline,lr.trace_wall,rtol=0,atol=1e-15)
assert len(events)==len(CONFIG.seeds)*CONFIG.epochs
accepted=events.loc[events.applied]
assert (accepted.probe_loss_after<=accepted.probe_loss_before+TRACE_WALL.loss_tolerance).all()
assert (result.corrections.projection_identity_error<5e-5).all()
assert set(result.spectral.weightwatcher_status)=={"ok"}
expected=[RUN_DIR/n for n in ["performance_by_epoch_and_seed.csv","spectral_metrics_by_epoch_layer_and_seed.csv","trace_wall_corrections_by_step_layer_and_seed.csv","performance_summary_95ci.csv","spectral_summary_95ci.csv","trace_wall_correction_summary.csv","config.json","paired_manifest.json"]]
for seed in CONFIG.seeds:
    d=RUN_DIR/"seeds"/f"seed_{seed}"; expected += [d/"baseline_final_state.pt",d/"trace_wall_final_state.pt"]
    for epoch in range(1,CONFIG.epochs+1): expected += [d/"checkpoints"/f"baseline_epoch_{epoch:03d}.pt",d/"checkpoints"/f"trace_wall_epoch_{epoch:03d}.pt"]
missing=[p for p in expected if not p.is_file()]
if missing: raise RuntimeError("Missing artifacts:\n"+"\n".join(map(str,missing)))
print("Audit passed: paired initialization/order/schedule, rotating training probes, loss-decreasing ECS corrections, WeightWatcher grids, and checkpoints.")